# Solver walkthrough

This notebook is a companion to `solver.py`. It is not meant to replace reading the source file; instead, it explains the main modelling choices and prints selected source functions for inspection.


In [ ]:
from pathlib import Path
import inspect
import pandas as pd

from solver import (
    SolveOptions, normalize_crossing_diagonal_mode,
    merge_crossing_truss_splits_to_continuous,
    prepare_model_for_crossing_diagonal_mode,
    determine_restrained_dofs, apply_truss_only_translation_stabilization,
    assemble_global_stiffness, solve_model, member_forces_dataframe,
    export_results_to_excel,
)

## 1. Crossing-diagonal modes

The same CSV files can be interpreted in three ways:

1. `continuous`: crossing diagonal segments are merged back into continuous pass-through trusses; generated crossing nodes are removed from the analysis.
2. `connected_stabilized`: crossing diagonals share the generated crossing node; tiny regularizing springs are added only if a truss-only node has missing translational stiffness directions.
3. `connected_unstabilized`: crossing diagonals share the generated crossing node, but no regularizing springs are added. This is useful for proving whether the model is rank deficient.


In [ ]:
print(inspect.getsource(normalize_crossing_diagonal_mode))
print(inspect.getsource(merge_crossing_truss_splits_to_continuous))
print(inspect.getsource(prepare_model_for_crossing_diagonal_mode))

## 2. Boundary conditions

The solver restrains:

- all nodes at minimum Z, if `fix_min_z_nodes=True`;
- all nodes explicitly marked `is_fixed_support=True`;
- rotations at truss-only nodes, because truss elements do not use rotational DOFs.

Restraining truss-only rotations does not add physical translational stiffness. It merely removes unused rotational unknowns.


In [ ]:
print(inspect.getsource(determine_restrained_dofs))

## 3. Truss-only stabilization

This is the deliberately small numerical regularization used only in `connected_stabilized` mode.

It identifies missing translational stiffness directions at truss-only nodes and adds tiny springs in those directions. The default is `1e-3 N/mm`, which is negligible compared with ordinary member axial stiffnesses.


In [ ]:
print(inspect.getsource(apply_truss_only_translation_stabilization))

## 4. Assembly and solving

The frame and truss elements are assembled into the same 6-DOF-per-node global matrix.

- Frame elements populate all translational and rotational DOFs.
- Truss elements populate only translational DOFs.


In [ ]:
print(inspect.getsource(assemble_global_stiffness))
print(inspect.getsource(solve_model))

## 5. Force recovery and signed stress output

The solver reports both absolute stress measures and signed tension/compression measures:

- `sigma_max_abs_MPa`: absolute maximum stress magnitude;
- `sigma_axial_signed_MPa`: signed axial stress;
- `sigma_extreme_signed_MPa`: signed controlling extreme-fiber stress for frames, signed axial stress for trusses.


In [ ]:
print(inspect.getsource(member_forces_dataframe))

## 6. Workbook export

The XLSX export function writes summary, critical ranked, critical matched, all-elements alternating, and per-model result sheets.


In [ ]:
print(inspect.getsource(export_results_to_excel))